# Phan tich va khuyen nghi AI Agent trong nganh Khoa hoc may tinh

Notebook nay doc 4 file CSV, loc cac nghe lien quan den CS/IT/Data/Software, noi du lieu theo `Task ID` va `User ID`, tinh chi so **AI Agent Readiness Score**, sau do tao bang va bieu do de dua ra khuyen nghi trien khai AI agent.

Cac file dau vao can nam cung thu muc voi notebook:
- `task_statement_with_metadata.csv`
- `expert_rated_technological_capability.csv`
- `domain_worker_desires.csv`
- `domain_worker_metadata.csv`

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    sns = None

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 14

DATA_DIR = Path(".")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    "tasks": DATA_DIR / "task_statement_with_metadata.csv",
    "expert": DATA_DIR / "expert_rated_technological_capability.csv",
    "desires": DATA_DIR / "domain_worker_desires.csv",
    "workers": DATA_DIR / "domain_worker_metadata.csv",
}

missing_files = [str(path) for path in FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Thieu file dau vao: {missing_files}")

## 1. Doc du lieu va xem tong quan

In [2]:
tasks = pd.read_csv(FILES["tasks"])
expert = pd.read_csv(FILES["expert"])
desires = pd.read_csv(FILES["desires"])
workers = pd.read_csv(FILES["workers"])

datasets = {
    "task_statement_with_metadata": tasks,
    "expert_rated_technological_capability": expert,
    "domain_worker_desires": desires,
    "domain_worker_metadata": workers,
}

overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
    }
    for name, df in datasets.items()
])
overview

,dataset,rows,columns,missing_cells
0,task_statement_with_metadata,2131,14,1284
1,expert_rated_technological_capability,2057,11,0
2,domain_worker_desires,5731,31,6544
3,domain_worker_metadata,1500,26,3250


In [3]:
for name, df in datasets.items():
    print("\n" + "=" * 90)
    print(name)
    print("Rows, Columns:", df.shape)
    print(df.dtypes)
    display(df.head(3))


task_statement_with_metadata
Rows, Columns: (2131, 14)
O*NET-SOC Code                                    object
Occupation (O*NET-SOC Title)                      object
Task ID                                            int64
Task                                              object
Task Type                                         object
Date                                              object
Category                                         float64
Frequency                                        float64
Importance                                       float64
Relevance                                        float64
Occupation Mean Annual Wage                      float64
Occupation Employment                            float64
Skill (O*NET Work Activity)                       object
Skill ID (O*NET Generalized Work Activity ID)     object
dtype: object


,O*NET-SOC Code,Occupation (O*NET-SOC Title),Task ID,Task,Task Type,Date,Category,Frequency,Importance,Relevance,Occupation Mean Annual Wage,Occupation Employment,Skill (O*NET Work Activity),Skill ID (O*NET Generalized Work Activity ID)
0,11-2011.00,Advertising and Promotions Managers,3226,"Inspect layouts and advertising copy, and edit scripts, audio, video, and other promotional material for adherence to specifications.",Core,08/2018,3.0,3.0,4.07,87.26,149270.0,21100.0,['Evaluating Information to Determine Compliance with Standards'],['4.A.2.a.3']
1,11-2011.00,Advertising and Promotions Managers,3242,Coordinate with the media to disseminate advertising.,Core,08/2018,3.0,3.0,3.87,80.19,149270.0,21100.0,"['Communicating with Supervisors, Peers, or Subordinates']",['4.A.4.a.2']
2,11-2011.00,Advertising and Promotions Managers,3223,Prepare budgets and submit estimates for program costs as part of campaign plan development.,Core,08/2018,3.0,3.0,3.67,81.73,149270.0,21100.0,"['Guiding, Directing, and Motivating Subordinates']",['4.A.4.b.4']



expert_rated_technological_capability
Rows, Columns: (2057, 11)
Task ID                                     int64
Occupation (O*NET-SOC Title)               object
Task                                       object
User ID                                    object
Date                                       object
Automation Capacity Rating                  int64
Physical Action Requirement                 int64
Involved Uncertainty                        int64
Domain Expertise Requirement                int64
Interpersonal Communication Requirement     int64
Human Agency Scale Rating                   int64
dtype: object


,Task ID,Occupation (O*NET-SOC Title),Task,User ID,Date,Automation Capacity Rating,Physical Action Requirement,Involved Uncertainty,Domain Expertise Requirement,Interpersonal Communication Requirement,Human Agency Scale Rating
0,3226,Advertising and Promotions Managers,"Inspect layouts and advertising copy, and edit scripts, audio, video, and other promotional material for adherence to specifications.",RedTiger,2025/3/16,4,1,1,2,1,2
1,21567,Appraisers and Assessors of Real Estate,Verify legal descriptions of properties by comparing them to county records.,YellowZebra,2025/3/16,3,1,2,4,1,2
2,3970,Technical Writers,"Review published materials and recommend revisions or changes in scope, format, content, and methods of reproduction and binding.",BrownFox,2025/3/18,4,3,3,3,3,3



domain_worker_desires
Rows, Columns: (5731, 31)
Task ID                                          int64
Occupation (O*NET-SOC Title)                    object
Task                                            object
User ID                                         object
Date                                            object
Self-reported Expertise                         object
Automation Desire Rating                         int64
Time                                             int64
Core Skill Rating                                int64
Job Security Rating                              int64
Enjoyment Rating                                 int64
Reasons for Automation Desire - Free Time         bool
Reasons for Automation Desire - Repetitive        bool
Reasons for Automation Desire - Human Error       bool
Reasons for Automation Desire - Stress            bool
Reasons for Automation Desire - Difficulty        bool
Reasons for Automation Desire - Scale             bool
Physical Action 

,Task ID,Occupation (O*NET-SOC Title),Task,User ID,Date,Self-reported Expertise,Automation Desire Rating,Time,Core Skill Rating,Job Security Rating,Enjoyment Rating,Reasons for Automation Desire - Free Time,Reasons for Automation Desire - Repetitive,Reasons for Automation Desire - Human Error,Reasons for Automation Desire - Stress,Reasons for Automation Desire - Difficulty,Reasons for Automation Desire - Scale,Physical Action Requirement,Interpersonal Communication Requirement,Involved Uncertainty,Domain Expertise Requirement,Human Agency Scale Rating,Reasons for Human Agency - Physical,Reasons for Human Agency - Control,Reasons for Human Agency - Domain Knowledge,Reasons for Human Agency - Empathy,Reasons for Human Agency - Quality Oversight,Reasons for Human Agency - Dynamic,Reasons for Human Agency - Ethical,Other Reason for Automation Desire,Other Reason for Human Agency
0,2587,Customer Service Representatives,Review insurance policy terms to determine whether a particular loss is covered by insurance.,9215f6fc-81e2-40e1-96db-3ac70cab4091,2025/2/25,Average,1,1,1,1,1,False,False,False,False,False,False,5,5,2,5,5,True,True,True,True,True,True,True,FALSE,NaN
1,2591,Customer Service Representatives,"Recommend improvements in products, packaging, shipping, service, or billing methods and procedures to prevent future problems.",9215f6fc-81e2-40e1-96db-3ac70cab4091,2025/2/25,Average,1,5,5,5,2,False,False,False,False,False,False,5,5,5,5,5,True,True,True,True,True,True,True,FALSE,NaN
2,55,Medical and Health Services Managers,"Maintain awareness of advances in medicine, computerized diagnostic and treatment equipment, data processing technology, government regulations, health insu...",a17b322e-3134-475c-a652-89f27b3a9946,2025/1/21,Expert,2,1,4,1,3,False,False,False,False,False,False,2,2,2,5,3,False,False,True,False,True,False,True,FALSE,NaN



domain_worker_metadata
Rows, Columns: (1500, 26)
User ID                                   object
Occupation (O*NET-SOC Title)              object
Gender                                    object
Race                                      object
Income                                    object
Age                                        int64
Education                                 object
Experience                                object
AI Tedious Work Attitude                  object
AI Job Importance Attitude                object
AI Daily Interest Attitude                object
AI Suffering Attitude                     object
Zip Code                                  object
Political Affiliation                     object
LLM Familiarity                           object
LLM Use in Work                           object
LLM Usage by Type - Information Access    object
LLM Usage by Type - Edit                  object
LLM Usage by Type - Idea Generation       object
LLM Usage by Type -

,User ID,Occupation (O*NET-SOC Title),Gender,Race,Income,Age,Education,Experience,AI Tedious Work Attitude,AI Job Importance Attitude,AI Daily Interest Attitude,AI Suffering Attitude,Zip Code,Political Affiliation,LLM Familiarity,LLM Use in Work,LLM Usage by Type - Information Access,LLM Usage by Type - Edit,LLM Usage by Type - Idea Generation,LLM Usage by Type - Communication,LLM Usage by Type - Analysis,LLM Usage by Type - Decision,LLM Usage by Type - Coding,LLM Usage by Type - System Design,LLM Usage by Type - Data Processing,Recruitment Source
0,a17b322e-3134-475c-a652-89f27b3a9946,Medical and Health Services Managers,Female,White,60-86K,30,"Professional Degree (e.g., MD, JD)",1-2 year,Somewhat agree,Neither agree nor disagree,Somewhat agree,Somewhat disagree,NaN,Green Party,I use them regularly.,"Yes, I use them every week in my work.",Daily,Never,Never,Never,Weekly,Weekly,Never,Never,Monthly,Prolific
1,533a6bf0-9958-4892-8777-44108350b9be,Customer Service Representatives,Male,Asian American and Pacific Islander,86K-165K,28,Bachelor’s Degree,6-10 years,Somewhat agree,Somewhat agree,Somewhat agree,Neither agree nor disagree,10010,Republican,I use them regularly.,"Yes, I use them every week in my work.",Weekly,Weekly,Daily,Daily,Weekly,Daily,Weekly,Never,Weekly,Prolific
2,9f48da63-2e55-413d-a8b4-ec18e410eb36,Mechanical Engineers,Male,Black,30-60K,34,Bachelor’s Degree,6-10 years,Strongly agree,Strongly agree,Somewhat agree,Somewhat disagree,6438,Republican,I have some experience using them.,"Yes, I have used them occasionally for specific tasks.",Weekly,Daily,Daily,Daily,Daily,Daily,Never,Never,Never,Prolific


## 2. Loc cac nghe lien quan den Khoa hoc may tinh / IT / Data

Ta loc theo ten nghe O*NET. Bien `INCLUDE_DATA_SCIENCE` cho phep mo rong sang thong ke, bioinformatics, biostatistics vi day la cac mang gan voi khoa hoc may tinh ung dung va du lieu.

In [4]:
INCLUDE_DATA_SCIENCE = True

core_cs_pattern = re.compile(
    r"(software|web developer|web administrator|database|information technology|"
    r"information security|computer programmer|computer systems|computer user support|"
    r"computer network|network and computer systems|computer and information)",
    flags=re.IGNORECASE,
)

data_science_pattern = re.compile(
    r"(statistician|biostatistician|bioinformatics)",
    flags=re.IGNORECASE,
)

exclude_pattern = re.compile(
    r"(computer numerically controlled|automated teller|office machine repairer|electronics engineers, except computer)",
    flags=re.IGNORECASE,
)

def is_cs_occupation(value):
    text = str(value)
    if exclude_pattern.search(text):
        return False
    if core_cs_pattern.search(text):
        return True
    if INCLUDE_DATA_SCIENCE and data_science_pattern.search(text):
        return True
    return False

all_occupations = pd.Series(pd.concat([
    df["Occupation (O*NET-SOC Title)"]
    for df in [tasks, expert, desires, workers]
    if "Occupation (O*NET-SOC Title)" in df.columns
]).dropna().unique()).sort_values()

cs_occupations = all_occupations[all_occupations.apply(is_cs_occupation)].tolist()
cs_occupations

['Bioinformatics Scientists',
 'Bioinformatics Technicians',
 'Biostatisticians',
 'Computer Network Architects',
 'Computer Network Support Specialists',
 'Computer Programmers',
 'Computer Systems Analysts',
 'Computer Systems Engineers/Architects',
 'Computer User Support Specialists',
 'Computer and Information Research Scientists',
 'Computer and Information Systems Managers',
 'Database Administrators',
 'Database Architects',
 'Information Security Analysts',
 'Information Technology Project Managers',
 'Network and Computer Systems Administrators',
 'Software Quality Assurance Analysts and Testers',
 'Statisticians',
 'Web Administrators',
 'Web Developers']

In [5]:
tasks_cs = tasks[tasks["Occupation (O*NET-SOC Title)"].isin(cs_occupations)].copy()
expert_cs = expert[expert["Occupation (O*NET-SOC Title)"].isin(cs_occupations)].copy()
desires_cs = desires[desires["Occupation (O*NET-SOC Title)"].isin(cs_occupations)].copy()
workers_cs = workers[workers["Occupation (O*NET-SOC Title)"].isin(cs_occupations)].copy()

pd.DataFrame({
    "table": ["tasks_cs", "expert_cs", "desires_cs", "workers_cs"],
    "rows": [len(tasks_cs), len(expert_cs), len(desires_cs), len(workers_cs)],
    "unique_occupations": [
        tasks_cs["Occupation (O*NET-SOC Title)"].nunique(),
        expert_cs["Occupation (O*NET-SOC Title)"].nunique(),
        desires_cs["Occupation (O*NET-SOC Title)"].nunique(),
        workers_cs["Occupation (O*NET-SOC Title)"].nunique(),
    ],
})

,table,rows,unique_occupations
0,tasks_cs,187,20
1,expert_cs,363,17
2,desires_cs,1116,17
3,workers_cs,286,17


## 3. Chuan hoa cot diem va noi du lieu theo Task ID

In [6]:
rating_cols_expert = [
    "Automation Capacity Rating",
    "Physical Action Requirement",
    "Involved Uncertainty",
    "Domain Expertise Requirement",
    "Interpersonal Communication Requirement",
    "Human Agency Scale Rating",
]

rating_cols_desires = [
    "Automation Desire Rating",
    "Time",
    "Core Skill Rating",
    "Job Security Rating",
    "Enjoyment Rating",
    "Physical Action Requirement",
    "Interpersonal Communication Requirement",
    "Involved Uncertainty",
    "Domain Expertise Requirement",
    "Human Agency Scale Rating",
]

for col in rating_cols_expert:
    expert_cs[col] = pd.to_numeric(expert_cs[col], errors="coerce")

for col in rating_cols_desires:
    desires_cs[col] = pd.to_numeric(desires_cs[col], errors="coerce")

for col in ["Importance", "Relevance", "Occupation Mean Annual Wage", "Occupation Employment"]:
    tasks_cs[col] = pd.to_numeric(tasks_cs[col], errors="coerce")

expert_task = expert_cs.groupby("Task ID", as_index=False).agg(
    expert_n=("Task ID", "size"),
    automation_capacity=("Automation Capacity Rating", "mean"),
    expert_physical=("Physical Action Requirement", "mean"),
    expert_uncertainty=("Involved Uncertainty", "mean"),
    expert_domain_expertise=("Domain Expertise Requirement", "mean"),
    expert_interpersonal=("Interpersonal Communication Requirement", "mean"),
    expert_human_agency=("Human Agency Scale Rating", "mean"),
)

desire_task = desires_cs.groupby("Task ID", as_index=False).agg(
    worker_n=("Task ID", "size"),
    automation_desire=("Automation Desire Rating", "mean"),
    time_required=("Time", "mean"),
    core_skill=("Core Skill Rating", "mean"),
    job_security=("Job Security Rating", "mean"),
    enjoyment=("Enjoyment Rating", "mean"),
    worker_physical=("Physical Action Requirement", "mean"),
    worker_uncertainty=("Involved Uncertainty", "mean"),
    worker_domain_expertise=("Domain Expertise Requirement", "mean"),
    worker_interpersonal=("Interpersonal Communication Requirement", "mean"),
    worker_human_agency=("Human Agency Scale Rating", "mean"),
)

task_model = (
    tasks_cs
    .merge(expert_task, on="Task ID", how="left")
    .merge(desire_task, on="Task ID", how="left")
)

task_model.head()

,O*NET-SOC Code,Occupation (O*NET-SOC Title),Task ID,Task,Task Type,Date,Category,Frequency,Importance,Relevance,Occupation Mean Annual Wage,Occupation Employment,Skill (O*NET Work Activity),Skill ID (O*NET Generalized Work Activity ID),expert_n,automation_capacity,expert_physical,expert_uncertainty,expert_domain_expertise,expert_interpersonal,expert_human_agency,worker_n,automation_desire,time_required,core_skill,job_security,enjoyment,worker_physical,worker_uncertainty,worker_domain_expertise,worker_interpersonal,worker_human_agency
0,11-3021.00,Computer and Information Systems Managers,978,Review project plans to plan and coordinate project activity.,Core,07/2016,3.0,3.0,4.08,99.97,187990.0,645970.0,"['Guiding, Directing, and Motivating Subordinates']",['4.A.4.b.4'],2.0,2.0,1.0,2.5,3.5,4.0,4.0,10.0,3.2,2.2,2.6,2.0,2.5,2.1,2.7,3.4,3.3,3.0
1,11-3021.00,Computer and Information Systems Managers,969,"Assign and review the work of systems analysts, programmers, and other computer-related workers.",Core,07/2016,5.0,5.0,4.04,92.97,187990.0,645970.0,"['Judging the Qualities of Objects, Services, or People']",['4.A.2.a.1'],2.0,2.0,1.5,2.0,2.5,2.5,3.5,10.0,3.5,2.6,3.7,2.0,3.3,2.1,2.7,3.0,2.9,2.4
2,11-3021.00,Computer and Information Systems Managers,971,"Develop computer information resources, providing for data security and control, strategic computing, and disaster recovery.",Core,07/2016,3.0,3.0,3.94,81.61,187990.0,645970.0,['Thinking Creatively'],['4.A.2.b.2'],2.0,3.0,1.5,2.5,3.5,1.5,2.5,10.0,3.6,2.9,3.3,2.4,3.1,2.2,3.1,3.7,3.1,2.8
3,11-3021.00,Computer and Information Systems Managers,970,Stay abreast of advances in technology.,Core,07/2016,4.0,4.0,3.88,100.00,187990.0,645970.0,['Updating and Using Relevant Knowledge'],['4.A.2.b.3'],2.0,2.5,1.0,1.0,4.0,1.0,3.5,10.0,3.3,1.7,2.8,1.7,2.9,1.6,2.5,2.8,1.9,2.3
4,11-3021.00,Computer and Information Systems Managers,972,Review and approve all systems charts and programs prior to their implementation.,Core,07/2016,3.0,3.0,3.68,87.19,187990.0,645970.0,"['Getting Information', 'Judging the Qualities of Objects, Services, or People']","['4.A.1.a.1', '4.A.2.a.1']",2.0,2.5,2.0,2.0,2.5,1.5,3.0,10.0,2.7,2.8,3.9,2.6,3.5,2.8,3.2,3.5,3.4,2.8


## 4. Tinh AI Agent Readiness Score

Cong thuc goi y:

`readiness = 0.30 * automation_capacity + 0.20 * automation_desire + 0.15 * (6 - human_agency) + 0.10 * (6 - physical) + 0.10 * (6 - interpersonal) + 0.10 * (6 - uncertainty) + 0.05 * task_importance`

Y nghia:
- Diem cao: phu hop de uu tien AI agent.
- Diem trung binh: nen dung AI copilot/human-in-the-loop.
- Diem thap: chua nen tu dong hoa manh.

In [ ]:
task_model["physical_requirement"] = task_model[["expert_physical", "worker_physical"]].mean(axis=1)
task_model["uncertainty"] = task_model[["expert_uncertainty", "worker_uncertainty"]].mean(axis=1)
task_model["domain_expertise"] = task_model[["expert_domain_expertise", "worker_domain_expertise"]].mean(axis=1)
task_model["interpersonal_requirement"] = task_model[["expert_interpersonal", "worker_interpersonal"]].mean(axis=1)
task_model["human_agency"] = task_model[["expert_human_agency", "worker_human_agency"]].mean(axis=1)

score_components = [
    "automation_capacity",
    "automation_desire",
    "human_agency",
    "physical_requirement",
    "interpersonal_requirement",
    "uncertainty",
    "Importance",
]

task_model_scored = task_model.dropna(subset=["automation_capacity", "automation_desire"], how="all").copy()

task_model_scored["ai_agent_readiness"] = (
    0.30 * task_model_scored["automation_capacity"]
    + 0.20 * task_model_scored["automation_desire"]
    + 0.15 * (6 - task_model_scored["human_agency"])
    + 0.10 * (6 - task_model_scored["physical_requirement"])
    + 0.10 * (6 - task_model_scored["interpersonal_requirement"])
    + 0.10 * (6 - task_model_scored["uncertainty"])
    + 0.05 * task_model_scored["Importance"]
)

task_model_scored["recommendation_type"] = np.select(
    [
        (task_model_scored["ai_agent_readiness"] >= 4.0) & (task_model_scored["human_agency"] <= 2.5),
        (task_model_scored["ai_agent_readiness"] >= 3.3) & (task_model_scored["human_agency"] <= 3.3),
        (task_model_scored["human_agency"] > 3.3) | (task_model_scored["domain_expertise"] >= 3.8) | (task_model_scored["uncertainty"] >= 3.5),
    ],
    [
        "Autonomous agent candidate",
        "Copilot / human-in-the-loop",
        "Human-led with AI assistance",
    ],
    default="Low priority / observe",
)

task_model_scored[[
    "Occupation (O*NET-SOC Title)", "Task ID", "Task", "ai_agent_readiness",
    "automation_capacity", "automation_desire", "human_agency", "domain_expertise",
    "uncertainty", "recommendation_type"
]].sort_values("ai_agent_readiness", ascending=False).head(15)

## 5. Bang ket qua chinh

In [ ]:
top_tasks = task_model_scored.sort_values("ai_agent_readiness", ascending=False)[[
    "Task ID",
    "Occupation (O*NET-SOC Title)",
    "Task",
    "ai_agent_readiness",
    "automation_capacity",
    "automation_desire",
    "human_agency",
    "domain_expertise",
    "uncertainty",
    "Importance",
    "Relevance",
    "Occupation Mean Annual Wage",
    "Occupation Employment",
    "recommendation_type",
]].copy()

top_tasks.head(20)

In [ ]:
occupation_summary = (
    task_model_scored
    .groupby("Occupation (O*NET-SOC Title)", as_index=False)
    .agg(
        tasks=("Task ID", "nunique"),
        readiness_mean=("ai_agent_readiness", "mean"),
        automation_capacity_mean=("automation_capacity", "mean"),
        automation_desire_mean=("automation_desire", "mean"),
        human_agency_mean=("human_agency", "mean"),
        domain_expertise_mean=("domain_expertise", "mean"),
        uncertainty_mean=("uncertainty", "mean"),
        importance_mean=("Importance", "mean"),
        relevance_mean=("Relevance", "mean"),
        employment_mean=("Occupation Employment", "mean"),
        wage_mean=("Occupation Mean Annual Wage", "mean"),
    )
    .sort_values("readiness_mean", ascending=False)
)

occupation_summary

## 6. Truc quan hoa

In [ ]:
plot_df = occupation_summary.sort_values("readiness_mean", ascending=True)

plt.figure(figsize=(11, max(6, len(plot_df) * 0.35)))
if sns:
    sns.barplot(data=plot_df, x="readiness_mean", y="Occupation (O*NET-SOC Title)", color="#4C78A8")
else:
    plt.barh(plot_df["Occupation (O*NET-SOC Title)"], plot_df["readiness_mean"], color="#4C78A8")
plt.title("AI Agent Readiness theo nghe CS/IT/Data")
plt.xlabel("Readiness score")
plt.ylabel("")
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "readiness_by_occupation.png", dpi=160)
plt.show()

In [ ]:
scatter_df = task_model_scored.dropna(subset=["automation_capacity", "human_agency", "automation_desire"])

plt.figure(figsize=(10, 7))
if sns:
    sns.scatterplot(
        data=scatter_df,
        x="automation_capacity",
        y="human_agency",
        size="automation_desire",
        hue="recommendation_type",
        sizes=(40, 240),
        alpha=0.75,
    )
else:
    plt.scatter(
        scatter_df["automation_capacity"],
        scatter_df["human_agency"],
        s=scatter_df["automation_desire"].fillna(2) * 35,
        alpha=0.7,
    )
plt.axvline(3.5, color="gray", linestyle="--", linewidth=1)
plt.axhline(3.0, color="gray", linestyle="--", linewidth=1)
plt.title("Automation Capacity vs Human Agency")
plt.xlabel("Automation Capacity Rating")
plt.ylabel("Human Agency Scale Rating")
plt.xlim(1, 5.2)
plt.ylim(1, 5.2)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "automation_vs_human_agency.png", dpi=160)
plt.show()

In [ ]:
reason_auto_cols = [c for c in desires_cs.columns if c.startswith("Reasons for Automation Desire - ")]
reason_human_cols = [c for c in desires_cs.columns if c.startswith("Reasons for Human Agency - ")]

auto_reason_rates = (
    desires_cs[reason_auto_cols]
    .replace({True: 1, False: 0, "True": 1, "False": 0})
    .apply(pd.to_numeric, errors="coerce")
    .mean()
    .mul(100)
    .sort_values(ascending=True)
)
auto_reason_rates.index = auto_reason_rates.index.str.replace("Reasons for Automation Desire - ", "", regex=False)

human_reason_rates = (
    desires_cs[reason_human_cols]
    .replace({True: 1, False: 0, "True": 1, "False": 0})
    .apply(pd.to_numeric, errors="coerce")
    .mean()
    .mul(100)
    .sort_values(ascending=True)
)
human_reason_rates.index = human_reason_rates.index.str.replace("Reasons for Human Agency - ", "", regex=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
auto_reason_rates.plot(kind="barh", ax=axes[0], color="#59A14F")
axes[0].set_title("Ly do muon tu dong hoa")
axes[0].set_xlabel("Ty le worker chon (%)")
human_reason_rates.plot(kind="barh", ax=axes[1], color="#E15759")
axes[1].set_title("Ly do can con nguoi tham gia")
axes[1].set_xlabel("Ty le worker chon (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "worker_reason_rates.png", dpi=160)
plt.show()

pd.DataFrame({
    "automation_reason_rate_percent": auto_reason_rates.sort_values(ascending=False),
}).join(pd.DataFrame({
    "human_agency_reason_rate_percent": human_reason_rates.sort_values(ascending=False),
}), how="outer")

## 7. Muc do san sang dung LLM cua worker trong nhom CS/IT/Data

In [ ]:
llm_cols = [
    "LLM Familiarity",
    "LLM Use in Work",
    "LLM Usage by Type - Coding",
    "LLM Usage by Type - System Design",
    "LLM Usage by Type - Analysis",
    "LLM Usage by Type - Data Processing",
]

for col in llm_cols:
    print("\n" + col)
    display(workers_cs[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count"))

In [ ]:
usage_cols = [
    "LLM Usage by Type - Coding",
    "LLM Usage by Type - System Design",
    "LLM Usage by Type - Analysis",
    "LLM Usage by Type - Data Processing",
    "LLM Usage by Type - Information Access",
    "LLM Usage by Type - Edit",
    "LLM Usage by Type - Idea Generation",
    "LLM Usage by Type - Communication",
]

usage_long = workers_cs[usage_cols].melt(var_name="usage_type", value_name="frequency")
usage_long["usage_type"] = usage_long["usage_type"].str.replace("LLM Usage by Type - ", "", regex=False)

usage_order = ["Never", "Monthly", "Weekly", "Daily"]
usage_share = (
    usage_long.dropna()
    .groupby(["usage_type", "frequency"])
    .size()
    .groupby(level=0)
    .apply(lambda s: 100 * s / s.sum())
    .rename("percent")
    .reset_index()
)

pivot_usage = usage_share.pivot(index="usage_type", columns="frequency", values="percent").fillna(0)
pivot_usage = pivot_usage[[c for c in usage_order if c in pivot_usage.columns]]
pivot_usage.plot(kind="barh", stacked=True, figsize=(12, 7), colormap="viridis")
plt.title("Tan suat dung LLM theo loai tac vu trong nhom CS/IT/Data")
plt.xlabel("Ty le (%)")
plt.ylabel("")
plt.legend(title="Frequency", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "llm_usage_by_type.png", dpi=160)
plt.show()

pivot_usage.round(1)

## 8. Bang khuyen nghi cuoi cung

In [ ]:
recommendation_table = top_tasks.copy()
recommendation_table["suggested_agent"] = np.select(
    [
        recommendation_table["Task"].str.contains("back up|backup|recovery|monitor|record|document", case=False, na=False),
        recommendation_table["Task"].str.contains("test|defect|compatibility|quality", case=False, na=False),
        recommendation_table["Task"].str.contains("analyze|statistical|data|report|chart|graph", case=False, na=False),
        recommendation_table["Task"].str.contains("configure|install|network|server|security|virus", case=False, na=False),
        recommendation_table["Task"].str.contains("plan|risk|project|collaborate|advise|design", case=False, na=False),
    ],
    [
        "Operations automation agent",
        "QA testing agent",
        "Data analysis / reporting agent",
        "IT monitoring and support agent",
        "Technical copilot with human approval",
    ],
    default="General task copilot",
)

recommendation_table["implementation_note"] = np.select(
    [
        recommendation_table["recommendation_type"].eq("Autonomous agent candidate"),
        recommendation_table["recommendation_type"].eq("Copilot / human-in-the-loop"),
        recommendation_table["recommendation_type"].eq("Human-led with AI assistance"),
    ],
    [
        "Co the tu dong hoa cao, nhung van can logging, rollback va canh bao loi.",
        "Agent de xuat hanh dong; con nguoi phe duyet truoc khi thuc thi.",
        "AI chi nen ho tro tong hop, checklist, phan tich so bo; con nguoi quyet dinh.",
    ],
    default="Theo doi them va chi thu nghiem o muc do nho.",
)

recommendation_table.head(25)

In [ ]:
task_model_scored.to_csv(OUTPUT_DIR / "ai_agent_task_scored.csv", index=False, encoding="utf-8-sig")
occupation_summary.to_csv(OUTPUT_DIR / "ai_agent_occupation_summary.csv", index=False, encoding="utf-8-sig")
recommendation_table.to_csv(OUTPUT_DIR / "ai_agent_recommendations.csv", index=False, encoding="utf-8-sig")

print("Da xuat:")
print(OUTPUT_DIR / "ai_agent_task_scored.csv")
print(OUTPUT_DIR / "ai_agent_occupation_summary.csv")
print(OUTPUT_DIR / "ai_agent_recommendations.csv")

## 9. Goi y ket luan cho bao cao

Mot ket luan co the dung:

> AI agent trong nganh khoa hoc may tinh nen duoc uu tien trien khai o cac tac vu co tinh lap lai, du lieu ro rang va rui ro thap nhu backup, monitoring, documentation, QA defect tracking, phan tich log/network va tao bao cao. Voi cac tac vu lien quan den thiet ke he thong, quan ly du an, an ninh thong tin, danh gia rui ro va quyet dinh kien truc, AI agent nen dong vai tro copilot co su phe duyet cua con nguoi.

Ban co the dung cac bang `top_tasks`, `occupation_summary`, va `recommendation_table` de viet phan ket qua va khuyen nghi.